# AML - LEAVE ONE VARIANT OUT 
This script submits the commands for the leave-one-variant-out (LOVO) analysis. It performs Step 2 of the SAIGE-GENE+ association tests for each variant set, where each set has exactly one variant removed from the full collection. The script iterates over all annotation files, with each file representing a reduced variant set missing one specific variant. By looping through these files, the pipeline re-runs Step 2 for every leave-one-out subset, allowing for evaluation of each variant’s individual contribution to the overall gene-based association signal and identifying variants that disproportionately influence the test statistic.

In [ ]:
# Imports the 'os' module, which allows interaction with the operating system (e.g., environment variables, file paths).
import os 

# Retrieves the value of the environment variable 'WORKSPACE_BUCKET'.
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Displays the value of 'my_bucket' (i.e., the workspace bucket URL or path).
my_bucket


# COMMAND 

In [ ]:
%%writefile lovo_high_confidence_analysis.sh

#!/bin/bash
set -o errexit
set -o nounset

annotation_list="${vat}/LOVO-annotation-chr${chrom}.txt"

for pop in EUR; do
  for pheno in aml_phenotype; do
   
    while IFS= read -r variant; do
        echo "Processing pop: ${pop} | phenotype: ${pheno} | chr: ${chrom} | variant: ${variant}"

        for annotation_table in ${vat}/LOVO-${variant}-annotation-*.txt; do
        
        echo "Processing annotation table: ${annotation_table}"
        
        step2_SPAtests.R \
            --bedFile="${plink_bed}" \
            --bimFile="${plink_bim}" \
            --famFile="${plink_fam}" \
            --SAIGEOutputFile="${output}/${pop}_${pheno}_lovo_high_confidence_${variant}" \
            --LOCO=FALSE \
            --AlleleOrder=alt-first \
            --minMAF=0 \
            --minMAC=0.5 \
            --GMMATmodelFile="${null_glmm}/${pop}_${pheno}_null_glmm.rda" \
            --varianceRatioFile="${null_glmm}/${pop}_${pheno}_null_glmm.varianceRatio.txt" \
            --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
            --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
            --groupFile="${annotation_table}" \
            --annotation_in_groupTest=AlphaMissense-0.564,LOFTEE-HC:SpliceAI-0.8,LOFTEE-HC:SpliceAI-0.8:AlphaMissense-0.564 \
            --maxMAF_in_groupTest=0.0001,0.001,0.01 \
            --is_output_markerList_in_groupTest=TRUE \
            --is_fastTest=TRUE

        done
    done < "${annotation_list}"

    done
done


In [ ]:
!gsutil cp lovo_high_confidence_analysis.sh {my_bucket}/data/scripts/

In [ ]:
!gsutil cat {my_bucket}/data/scripts/lovo_high_confidence_analysis.sh

# SUBMIT COMMAND 

In [ ]:
%%bash --out saige_gene_step02

LOWER=2
UPPER=3
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 20 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bed" \
  --input plink_bim="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bim" \
  --input plink_fam="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/EUR_LOVO_annotation/EUR_LOVO_annotation_chr${chromo}/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/LOVO_analysis/" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/lovo_high_confidence_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=4
UPPER=5
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 20 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bed" \
  --input plink_bim="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bim" \
  --input plink_fam="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/EUR_LOVO_annotation/EUR_LOVO_annotation_chr${chromo}/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/LOVO_analysis/" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/lovo_high_confidence_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=15
UPPER=16
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 20 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bed" \
  --input plink_bim="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bim" \
  --input plink_fam="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/EUR_LOVO_annotation/EUR_LOVO_annotation_chr${chromo}/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/LOVO_analysis/" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/lovo_high_confidence_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=17
UPPER=18
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 20 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bed" \
  --input plink_bim="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bim" \
  --input plink_fam="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/EUR_LOVO_annotation/EUR_LOVO_annotation_chr${chromo}/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/LOVO_analysis/" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/lovo_high_confidence_analysis.sh"

done

## DOWNLOAD RESULTS FROM THE WORKSPACE BUCKET 

In [ ]:
!rm -rf saige_gene_step02
!rm -rf merged_saige_gene_step02
!rm -rf markerList

In [ ]:
!mkdir -p saige_gene_step02
!mkdir -p merged_saige_gene_step02
!mkdir -p markerList 

In [ ]:
!gsutil cp {my_bucket}/data/saige_step02/LOVO_analysis/* saige_gene_step02
!gsutil cp {my_bucket}/data/saige_step02/HC_marklist/* markerList

In [ ]:
# Load packages 
import os
import glob
import pandas as pd

# ===============================================================================================
# Load files from 
# ===============================================================================================

files = glob.glob("saige_gene_step02/*_deprivation_index")
files = [f for f in files if ".index" not in f]
files = [f for f in files if ".singleAssoc" not in f]
files = [f for f in files if ".markerList" not in f]

dfs = []
for f in files:
    df = pd.read_csv(f, sep="\t", header=0)  
    df['filename'] = os.path.basename(f)    
    dfs.append(df)


combined_df = pd.concat(dfs, ignore_index=True)
combined_df["var"] = combined_df["filename"].str.split("_", expand = True)[6]
combined_df = combined_df.drop(columns = ['filename'])

# ================================================================================================
# Filter out variants not present in the rare and ultra-rare marker list files 
# ================================================================================================

ultra_rare_variants = pd.read_csv("EUR-AML-Ultra-Rare-Variants.markerList.txt", 
                                  sep = '\t')
rare_variants = pd.read_csv("EUR-AML-Rare-Variants.markerList.txt", 
                            sep = '\t')
combined_df = combined_df[(combined_df['var'].isin(ultra_rare_variants['Ultra_Rare_Variants'])) | 
                          (combined_df['var'].isin(rare_variants['Rare_Variants']))] 

# ================================================================================================
# Merge annotation file with leave-one-variant-out file 
# ================================================================================================

group_annotation = pd.read_csv("markerList/Group-Annotation.txt", sep = '\t')
combined_df = group_annotation.merge(combined_df, how = "inner", on =["var", 'Region'])

# ================================================================================================
# Merge all rare and ultra-rare AoU variants identified in Genomic Data Commons into a single list
# ================================================================================================

gdc_mutations = pd.read_csv("markerList/GDC-frequent-mutations.tsv", 
                            sep = '\t')
hgvs_ultra_rare = pd.read_csv("markerList/EUR-AML-Ultra-Rare-HGVS.markerList.txt", 
                              sep = '\t')
hgvs_rare = pd.read_csv("markerList/EUR-AML-Rare-HGVS.markerList.txt", 
                        sep = '\t')

hgvs_variants = pd.concat([hgvs_ultra_rare, hgvs_rare])
hgvs_variants["Variants"] = hgvs_variants.apply(
    lambda row: row["Ultra_Rare_Variants"] 
                if pd.notna(row["Ultra_Rare_Variants"]) 
                else row["Rare_Variants"],
    axis=1
) 

hgvs_variants = (
    hgvs_variants.merge(
        gdc_mutations.rename(columns={'dna_change':'HGVS'}), 
                    on = "HGVS", 
        how ="inner" )
)

# ================================================================================================
# Merge all rare and ultra-rare AoU variants identified in ClinVar into a single list
# ================================================================================================

clinvar_ultra_rare = pd.read_csv("markerList/EUR-AML-Ultra-Rare-Clinvar.markerList.txt", 
                                 sep = '\t')
clinvar_rare = pd.read_csv("markerList/EUR-AML-Rare-Clinvar.markerList.txt", 
                           sep = '\t')
clinvar_variants = pd.concat([clinvar_ultra_rare, clinvar_rare])

# ================================================================================================
# Label variants as either present or absent in the Genomic Data Commons or Clinvar 
# ================================================================================================

combined_df['Clinvar_Match'] = combined_df['var'].isin(clinvar_variants['Variants']).map(
    {True: "Match", False: "No Match"}
)
combined_df['GDC_Match'] = combined_df['var'].isin(hgvs_variants['Variants']).map(
    {True: "Match", False: "No Match"}
)
all_variants = set(clinvar_variants['Variants']) | set(hgvs_variants['Variants'])

combined_df['Match_Status'] = combined_df['var'].isin(all_variants).map(
    {True: "Match", False: "No Match"}
)

combined_df = combined_df[['var','anno','Region','Group','max_MAF',
                'Pvalue','Pvalue_Burden','Pvalue_SKAT',
                'BETA_Burden','SE_Burden',
                'Clinvar_Match','GDC_Match','Match_Status']]
# Save results 
combined_df.to_csv("EUR_aml_phenotype_leave_one_variant_out.txt", 
                   sep="\t", index=False)